In [1]:
import duckdb, boto3, os, requests, tarfile, gdown, gzip, time
import polars as pl
from pathlib import Path
import xml.etree.ElementTree as ET
os.makedirs('data', exist_ok=True)
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/xmls", exist_ok=True)
os.makedirs("data/parquet", exist_ok=True)

In [10]:
session = boto3.Session(profile_name='hiv-project')
s3 = session.client('s3')
bucket_name = 'hiv-data-022784797781'

**Please note**: the below link is a place-holder... you need to request access from HIVDB if you want the TCE dataset!

In [25]:
url = "https://drive.google.com/file/d/19PRBBOEFZGalIXE8am64L1e6dyBlF5K_/view?usp=sharing"
tar_path = "data/raw/tce_xmls.tar.gz"

In [ ]:
gdown.download(url, tar_path, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=19PRBBOEFZGalIXE8am64L1e6dyBlF5K_
To: /Users/benzenesea/Desktop/hiv-resistance-analytics/ingestion/data/raw/tce_xmls.tar.gz
100%|██████████| 24.8M/24.8M [00:04<00:00, 4.96MB/s]


In [22]:
import tarfile

with tarfile.open("data/raw/tce_xmls.tar.gz", "r:") as tar:
    tar.extractall(path="data/xmls", filter='tar')

---

Parse the XML to Parquet.

In [2]:
def parse_all(xml_path):
    root = ET.parse(xml_path).getroot()

    filename = Path(xml_path).name
    patient = root.findtext('patient/patientAlias')
    baseline_year = root.findtext('baselineYear')
    cd4_nadir = root.findtext('patient/CD4NadirBeforeTCE')
    date_unit = root.findtext('dateUnit')
    schema_version = root.findtext('schemaVersion')

    case_rows = [{
        "xml_filename": filename,
        "patient_alias": patient,
        "baseline_year": baseline_year,
        "cd4_nadir_before_tce": cd4_nadir,
        "date_unit": date_unit,
        "schema_version": schema_version
    }]

    measurements = []
    isolates = []
    mutations = []
    treatments = []

    # measurements
    for tag, mtype, val_fields in [
        ("baselineRNA", "RNA", ["logLoad", "rawLoad"]),
        ("pastRNA", "RNA", ["logLoad", "rawLoad"]),
        ("followupRNA", "RNA", ["logLoad", "rawLoad"]),
        ("baselineCD4", "CD4", ["count"]),
        ("pastCD4", "CD4", ["count"]),
        ("followupCD4", "CD4", ["count"]),
    ]:
        for node in root.findall(f".//{tag}"):
            for vf in val_fields:
                val = node.findtext(vf)
                if val:
                    measurements.append({
                        "xml_filename": filename,
                        "patient_alias": patient,
                        "relative_date": node.findtext("relativeDate"),
                        "measurement_type": mtype,
                        "timepoint_tag": tag,
                        "timepoint_type": tag.replace(mtype, "").lower(),
                        "value": val,
                        "value_type": vf
                    })

    # isolates + mutations
    for tag, itype in [("baselineIsolate", "baseline"), ("pastIsolate", "past")]:
        for i, iso in enumerate(root.findall(f".//{tag}")):
            iso_id = f"{filename}_{itype}_{i}"

            isolates.append({
                "xml_filename": filename,
                "patient_alias": patient,
                "isolate_id": iso_id,
                "isolate_type": itype,
                "gene": iso.findtext("gene"),
                "subtype": iso.findtext("subtype"),
                "relative_date": iso.findtext("relativeDate"),
                "aa_start": iso.findtext("aaStart"),
                "aa_stop": iso.findtext("aaStop"),
                "aa_sequence": iso.findtext("aaSequence"),
                "nucleotide_sequence": iso.findtext("nucleotideSequence"),
            })

            for mut in iso.findall(".//aaMutation"):
                mutations.append({
                    "xml_filename": filename,
                    "patient_alias": patient,
                    "isolate_id": iso_id,
                    "isolate_type": itype,
                    "gene": iso.findtext("gene"),
                    "relative_date": iso.findtext("relativeDate"),
                    "position": mut.findtext("position"),
                    "amino_acid": mut.findtext("aminoAcid"),
                    "mixtures": mut.findtext("mixtures"),
                })

    # treatments — NEW (baseline)
    for tnode in root.findall(".//baselineNewTreatment"):
        start = tnode.findtext("relativeStartDate")
        stop = tnode.findtext("relativeStopDate")
        duration = tnode.findtext("duration")
        for drug in tnode.findall("drug"):
            treatments.append({
                "xml_filename": filename,
                "patient_alias": patient,
                "relative_start_date": start,
                "relative_stop_date": stop,
                "duration": duration,
                "drug_code": drug.findtext("drugCode"),
                "drug_class": drug.findtext("drugClass"),
                "treatment_type": "new"
            })

    # treatments — ADD THIS (past)
    for tnode in root.findall(".//pastRegimenTreatments"):
        start = tnode.findtext("relativeStartDate")
        stop = tnode.findtext("relativeStopDate")
        duration = tnode.findtext("duration")
        for drug in tnode.findall("drug"):
            treatments.append({
                "xml_filename": filename,
                "patient_alias": patient,
                "relative_start_date": start,
                "relative_stop_date": stop,
                "duration": duration,
                "drug_code": drug.findtext("drugCode"),
                "drug_class": drug.findtext("drugClass"),
                "treatment_type": "past"
            })

    return case_rows, measurements, isolates, mutations, treatments

In [3]:
xml_files = list(Path("data/xmls/xmls_db").glob("*.xml"))

all_cases = []
all_measurements = []
all_isolates = []
all_mutations = []
all_treatments = []

for f in xml_files:
    try:
        case_rows, measurements, isolates, mutations, treatments = parse_all(f)
        all_cases.extend(case_rows)
        all_measurements.extend(measurements)
        all_isolates.extend(isolates)
        all_mutations.extend(mutations)
        all_treatments.extend(treatments)
    except Exception as e:
        print(f"Failed {f.name}: {e}")

df_case = pl.DataFrame(all_cases)
df_measurements = pl.DataFrame(all_measurements)
df_isolates = pl.DataFrame(all_isolates)
df_mutations = pl.DataFrame(all_mutations)
df_treatments = pl.DataFrame(all_treatments)

# measurements: partitioned by measurement_type, measurement_type must NOT stay in the file
for key, subdf in df_measurements.partition_by("measurement_type", as_dict=True).items():
    measurement_val = key[0] if isinstance(key, tuple) else key
    subdf = subdf.drop("measurement_type")
    subdf.write_parquet(
        f"data/parquet/tce_measurements/measurement_type={measurement_val}/data.parquet",
        mkdir=True
    )

# non-partitioned tables
df_case.write_parquet("data/parquet/tce_case/data.parquet", mkdir=True)
df_isolates.write_parquet("data/parquet/tce_isolates/data.parquet", mkdir=True)
df_mutations.write_parquet("data/parquet/tce_mutations/data.parquet", mkdir=True)
df_treatments.write_parquet("data/parquet/tce_treatments/data.parquet", mkdir=True)

In [4]:
df_case.head()

xml_filename,patient_alias,baseline_year,cd4_nadir_before_tce,date_unit,schema_version
str,str,str,str,str,str
"""162.xml""","""4303""","""2000""","""5""","""week""","""1.0"""
"""1390.xml""","""9659""","""2007""","""267""","""week""","""1.0"""
"""604.xml""","""14387""","""2006""","""296""","""week""","""1.0"""
"""88.xml""","""2075""","""2000""","""19""","""week""","""1.0"""
"""610.xml""","""14442""","""2003""","""106""","""week""","""1.0"""


In [5]:
df_isolates.head()

xml_filename,patient_alias,isolate_id,isolate_type,gene,subtype,relative_date,aa_start,aa_stop,aa_sequence,nucleotide_sequence
str,str,str,str,str,str,str,str,str,str,str
"""162.xml""","""4303""","""162.xml_baseline_0""","""baseline""","""PR""","""B""","""-8""","""1""","""99""","""PQITLWQRPIVTIKVGGQLREALLDTGADD…","""CCTCAAATCACTCTTTGGCAACGACCCATT…"
"""162.xml""","""4303""","""162.xml_baseline_1""","""baseline""","""RT""","""B""","""-8""","""1""","""304""","""PISPIETVPVKLKPGMDGPRVKQWPLTEEK…","""CCCATTAGTCCTATTGAAACTGTACCAGTA…"
"""162.xml""","""4303""","""162.xml_past_0""","""past""","""RT""","""B""","""-35""","""1""","""291""","""PISPIETVPVKLKPGMDGPRVKQWPLTEEK…","""CCCATTAGTCCTATTGAAACTGTACCAGTA…"
"""162.xml""","""4303""","""162.xml_past_1""","""past""","""PR""","""B""","""-35""","""1""","""99""","""PQITLWQRPIVTIKVGGQLREALLDTGADD…","""CCTCAAATCACTCTTTGGCAACGACCCATT…"
"""1390.xml""","""9659""","""1390.xml_baseline_0""","""baseline""","""PR""","""B""","""-10""","""1""","""99""","""PQITLWQRPLVPIRIGGQLKEALLDTGADD…","""CCTCARATCACTCTTTGGCAACGACCCCTC…"


In [6]:
df_measurements.head()

xml_filename,patient_alias,relative_date,measurement_type,timepoint_tag,timepoint_type,value,value_type
str,str,str,str,str,str,str,str
"""162.xml""","""4303""","""-8""","""RNA""","""baselineRNA""","""baseline""","""3.9""","""logLoad"""
"""162.xml""","""4303""","""-8""","""RNA""","""baselineRNA""","""baseline""","""7943""","""rawLoad"""
"""162.xml""","""4303""","""-166""","""RNA""","""pastRNA""","""past""","""3""","""logLoad"""
"""162.xml""","""4303""","""-166""","""RNA""","""pastRNA""","""past""","""1000""","""rawLoad"""
"""162.xml""","""4303""","""-158""","""RNA""","""pastRNA""","""past""","""4.2""","""logLoad"""


In [7]:
df_mutations.head()

xml_filename,patient_alias,isolate_id,isolate_type,gene,relative_date,position,amino_acid,mixtures
str,str,str,str,str,str,str,str,str
"""162.xml""","""4303""","""162.xml_baseline_0""","""baseline""","""PR""","""-8""","""10""","""I""",""""""
"""162.xml""","""4303""","""162.xml_baseline_0""","""baseline""","""PR""","""-8""","""15""","""V""",""""""
"""162.xml""","""4303""","""162.xml_baseline_0""","""baseline""","""PR""","""-8""","""20""","""R""",""""""
"""162.xml""","""4303""","""162.xml_baseline_0""","""baseline""","""PR""","""-8""","""35""","""D""",""""""
"""162.xml""","""4303""","""162.xml_baseline_0""","""baseline""","""PR""","""-8""","""36""","""I""",""""""


In [8]:
df_treatments.head()

xml_filename,patient_alias,relative_start_date,relative_stop_date,duration,drug_code,drug_class,treatment_type
str,str,str,str,str,str,str,str
"""162.xml""","""4303""","""0""","""44""","""43""","""ABC""","""NRTI""","""new"""
"""162.xml""","""4303""","""0""","""44""","""43""","""D4T""","""NRTI""","""new"""
"""162.xml""","""4303""","""0""","""44""","""43""","""3TC""","""NRTI""","""new"""
"""162.xml""","""4303""","""-28""","""0""","""27""","""ABC""","""NRTI""","""past"""
"""162.xml""","""4303""","""-28""","""0""","""27""","""EFV""","""NNRTI""","""past"""


---

Upload to S3.

In [11]:
def upload_to_s3(local_dir, bucket, prefix, s3_client):
    """Upload partitioned Parquet to S3."""
    local = Path(local_dir)
    for file in local.rglob('*.parquet'):
        key = f"{prefix}{file.relative_to(local)}"
        s3_client.upload_file(str(file), bucket, key)
        print(f"Uploaded: s3://{bucket}/{key}")

# Upload the partitioned datasets.
upload_to_s3('data/parquet/tce_measurements/', bucket_name, 'hivdb-tce/tce_measurements/', s3)
upload_to_s3('data/parquet/tce_case/', bucket_name, 'hivdb-tce/tce_case/', s3)
upload_to_s3('data/parquet/tce_isolates/', bucket_name, 'hivdb-tce/tce_isolates/', s3)
upload_to_s3('data/parquet/tce_mutations/', bucket_name, 'hivdb-tce/tce_mutations/', s3)
upload_to_s3('data/parquet/tce_treatments/', bucket_name, 'hivdb-tce/tce_treatments/', s3)

Uploaded: s3://hiv-data-022784797781/hivdb-tce/tce_measurements/measurement_type=RNA/data.parquet
Uploaded: s3://hiv-data-022784797781/hivdb-tce/tce_measurements/measurement_type=CD4/data.parquet
Uploaded: s3://hiv-data-022784797781/hivdb-tce/tce_case/data.parquet
Uploaded: s3://hiv-data-022784797781/hivdb-tce/tce_isolates/data.parquet
Uploaded: s3://hiv-data-022784797781/hivdb-tce/tce_mutations/data.parquet
Uploaded: s3://hiv-data-022784797781/hivdb-tce/tce_treatments/data.parquet


---

Use Glue to crawl the data.

In [273]:
def create_crawler(name, s3_path, database='hivdb_tce'):
    """Create Glue crawler for partitioned Parquet data."""
    glue = session.client('glue')
    
    # Get your account ID
    sts = session.client('sts')
    account_id = sts.get_caller_identity()['Account']
    
    # Use your account ID in the role ARN
    role_arn = f'arn:aws:iam::{account_id}:role/glue_crawler_role'
    
    glue.create_crawler(
        Name=name,
        Role=role_arn,
        DatabaseName=database,
        Targets={'S3Targets': [{'Path': s3_path}]},
        SchemaChangePolicy={
            'UpdateBehavior': 'UPDATE_IN_DATABASE',
            'DeleteBehavior': 'DEPRECATE_IN_DATABASE'
        }
    )
    
create_crawler('tce_measurements_crawler', f's3://{bucket_name}/hivdb-tce/tce_measurements/')
create_crawler('tce_case_crawler', f's3://{bucket_name}/hivdb-tce/tce_case/')
create_crawler('tce_isolates_crawler', f's3://{bucket_name}/hivdb-tce/tce_isolates/')
create_crawler('tce_mutations_crawler', f's3://{bucket_name}/hivdb-tce/tce_mutations/')

In [406]:
create_crawler('tce_treatments_crawler', f's3://{bucket_name}/hivdb-tce/tce_treatments/')

In [12]:
glue = session.client('glue')

In [13]:
glue.start_crawler(Name='tce_measurements_crawler')

{'ResponseMetadata': {'RequestId': 'f4a799b8-6863-4de6-bc17-c5a88995d498',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 18:56:27 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'f4a799b8-6863-4de6-bc17-c5a88995d498',
   'cache-control': 'no-cache'},
  'RetryAttempts': 0}}

In [14]:
glue.start_crawler(Name='tce_case_crawler')

{'ResponseMetadata': {'RequestId': '214749e5-d074-470e-a4c1-8e77d764e49b',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 18:56:28 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': '214749e5-d074-470e-a4c1-8e77d764e49b',
   'cache-control': 'no-cache'},
  'RetryAttempts': 0}}

In [15]:
glue.start_crawler(Name='tce_isolates_crawler')

{'ResponseMetadata': {'RequestId': '598bd937-7902-4ad6-8dc9-4e2e44a9155e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 18:56:30 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': '598bd937-7902-4ad6-8dc9-4e2e44a9155e',
   'cache-control': 'no-cache'},
  'RetryAttempts': 0}}

In [16]:
glue.start_crawler(Name='tce_mutations_crawler')

{'ResponseMetadata': {'RequestId': 'e17aef6c-c543-4131-86cb-32dd35e8a0e3',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 18:56:37 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'e17aef6c-c543-4131-86cb-32dd35e8a0e3',
   'cache-control': 'no-cache'},
  'RetryAttempts': 0}}

In [17]:
glue.start_crawler(Name='tce_treatments_crawler')

{'ResponseMetadata': {'RequestId': '12dcf22b-f202-44b3-ad69-d3fafc9a2bd3',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 18:56:38 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': '12dcf22b-f202-44b3-ad69-d3fafc9a2bd3',
   'cache-control': 'no-cache'},
  'RetryAttempts': 0}}

In [20]:
glue.get_crawler(Name='tce_treatments_crawler')['Crawler']['State']

'READY'

In [21]:
glue = session.client('glue')

table = glue.get_table(
    DatabaseName='hivdb_tce',
    Name='tce_measurements'
)

cols = table['Table']['StorageDescriptor']['Columns']
[col['Name'] for col in cols]

['xml_filename',
 'patient_alias',
 'relative_date',
 'timepoint_tag',
 'timepoint_type',
 'value',
 'value_type']

In [22]:
glue = session.client('glue')

table = glue.get_table(
    DatabaseName='hivdb_tce',
    Name='tce_case'
)

cols = table['Table']['StorageDescriptor']['Columns']
[col['Name'] for col in cols]

['xml_filename',
 'patient_alias',
 'baseline_year',
 'cd4_nadir_before_tce',
 'date_unit',
 'schema_version']

In [23]:
glue = session.client('glue')

table = glue.get_table(
    DatabaseName='hivdb_tce',
    Name='tce_isolates'
)

cols = table['Table']['StorageDescriptor']['Columns']
[col['Name'] for col in cols]

['xml_filename',
 'patient_alias',
 'isolate_id',
 'isolate_type',
 'gene',
 'subtype',
 'relative_date',
 'aa_start',
 'aa_stop',
 'aa_sequence',
 'nucleotide_sequence']

In [24]:
glue = session.client('glue')

table = glue.get_table(
    DatabaseName='hivdb_tce',
    Name='tce_mutations'
)

cols = table['Table']['StorageDescriptor']['Columns']
[col['Name'] for col in cols]

['xml_filename',
 'patient_alias',
 'isolate_id',
 'isolate_type',
 'gene',
 'relative_date',
 'position',
 'amino_acid',
 'mixtures']

In [25]:
glue = session.client('glue')

table = glue.get_table(
    DatabaseName='hivdb_tce',
    Name='tce_treatments'
)

cols = table['Table']['StorageDescriptor']['Columns']
[col['Name'] for col in cols]

['xml_filename',
 'patient_alias',
 'relative_start_date',
 'relative_stop_date',
 'duration',
 'drug_code',
 'drug_class',
 'treatment_type']

---

Athena query checking.

In [26]:
athena = boto3.client("athena", region_name="us-east-1")

In [27]:
def run_athena_query(sql, database="hivdb_tce", output="s3://hiv-data-022784797781/athena-results/"):
    resp = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": database},
        ResultConfiguration={"OutputLocation": output},
    )
    qid = resp["QueryExecutionId"]

    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)
        state = status["QueryExecution"]["Status"]["State"]

        if state == "SUCCEEDED":
            break
        if state in ("FAILED", "CANCELLED"):
            reason = status["QueryExecution"]["Status"].get("StateChangeReason", "Unknown error")
            raise RuntimeError(f"Athena query {state}: {reason}")

        time.sleep(1)

    results = athena.get_query_results(QueryExecutionId=qid)
    rows = results["ResultSet"]["Rows"]

    headers = [c["VarCharValue"] for c in rows[0]["Data"]]
    data = []
    for row in rows[1:]:
        values = [col.get("VarCharValue") for col in row.get("Data", [])]
        data.append(dict(zip(headers, values)))

    return data

#### Run the queries here:

Checking `dim_relative_time`.

In [29]:
rows = run_athena_query("""
    SELECT *
    FROM hivdb_tce.dim_relative_time
    LIMIT 10
""")

print(rows[:5])

[{'relative_time_key': '1', 'relative_date': '-1', 'date_unit': 'week', 'relative_weeks': '-1.0', 'timepoint_phase': 'pre', 'timepoint_bucket': 'recent_pre', 'is_baseline': 'false'}, {'relative_time_key': '2', 'relative_date': '-10', 'date_unit': 'week', 'relative_weeks': '-10.0', 'timepoint_phase': 'pre', 'timepoint_bucket': 'recent_pre', 'is_baseline': 'false'}, {'relative_time_key': '3', 'relative_date': '-100', 'date_unit': 'week', 'relative_weeks': '-100.0', 'timepoint_phase': 'pre', 'timepoint_bucket': 'far_pre', 'is_baseline': 'false'}, {'relative_time_key': '4', 'relative_date': '-101', 'date_unit': 'week', 'relative_weeks': '-101.0', 'timepoint_phase': 'pre', 'timepoint_bucket': 'far_pre', 'is_baseline': 'false'}, {'relative_time_key': '5', 'relative_date': '-102', 'date_unit': 'week', 'relative_weeks': '-102.0', 'timepoint_phase': 'pre', 'timepoint_bucket': 'far_pre', 'is_baseline': 'false'}]


In [ ]:
rows = run_athena_query("""
    SELECT *
    FROM hivdb_tce.dim_relative_time
    LIMIT 1
""")

print(rows[0].keys())

dict_keys(['relative_time_key', 'relative_date', 'date_unit', 'relative_weeks', 'timepoint_phase', 'timepoint_bucket', 'is_baseline'])


Checking `dim_regimen`.

In [31]:
rows = run_athena_query("""
    SELECT *
    FROM hivdb_tce.dim_regimen
    LIMIT 10
""")

print(rows[:5])

[{'regimen_key': '1', 'xml_filename': '1.xml', 'treatment_type': 'new', 'drug_count': '4', 'contains_pi': '1', 'contains_nrti': '1', 'contains_nnrti': '0', 'contains_insti': '0', 'regimen_signature': '3TC+D4T+IDV+RTV'}, {'regimen_key': '2', 'xml_filename': '1.xml', 'treatment_type': 'past', 'drug_count': '8', 'contains_pi': '1', 'contains_nrti': '1', 'contains_nnrti': '0', 'contains_insti': '0', 'regimen_signature': '3TC+AZT+D4T+DDI+IDV+NFV+RTV+SQV'}, {'regimen_key': '3', 'xml_filename': '10.xml', 'treatment_type': 'new', 'drug_count': '3', 'contains_pi': '1', 'contains_nrti': '1', 'contains_nnrti': '0', 'contains_insti': '0', 'regimen_signature': '3TC+D4T+NFV'}, {'regimen_key': '4', 'xml_filename': '10.xml', 'treatment_type': 'past', 'drug_count': '7', 'contains_pi': '1', 'contains_nrti': '1', 'contains_nnrti': '1', 'contains_insti': '0', 'regimen_signature': '3TC+AZT+D4T+DDI+NFV+NVP+SQV'}, {'regimen_key': '5', 'xml_filename': '100.xml', 'treatment_type': 'new', 'drug_count': '4', 'co

In [32]:
rows = run_athena_query("""
    SELECT *
    FROM hivdb_tce.dim_regimen
    LIMIT 1
""")

print(rows[0].keys())

dict_keys(['regimen_key', 'xml_filename', 'treatment_type', 'drug_count', 'contains_pi', 'contains_nrti', 'contains_nnrti', 'contains_insti', 'regimen_signature'])


Checking `dim_time`.

In [34]:
rows = run_athena_query("""
    SELECT *
    FROM hivdb_tce.dim_time
    LIMIT 10
""")

print(rows[:5])

[{'time_key': '1', 'baseline_year': '1997', 'decade': '1990s', 'year_bucket': '<2000'}, {'time_key': '2', 'baseline_year': '1998', 'decade': '1990s', 'year_bucket': '<2000'}, {'time_key': '3', 'baseline_year': '1999', 'decade': '1990s', 'year_bucket': '<2000'}, {'time_key': '4', 'baseline_year': '2000', 'decade': '2000s', 'year_bucket': '2000-2005'}, {'time_key': '5', 'baseline_year': '2001', 'decade': '2000s', 'year_bucket': '2000-2005'}]


In [35]:
rows = run_athena_query("""
    SELECT *
    FROM hivdb_tce.dim_time
    LIMIT 1
""")

print(rows[0].keys())

dict_keys(['time_key', 'baseline_year', 'decade', 'year_bucket'])


Checking `fact_tce`.

In [36]:
rows = run_athena_query("""
    SELECT *
    FROM hivdb_tce.fact_tce
    LIMIT 10
""")

print(rows[:5])

[{'tce_id': '1', 'xml_filename': '1.xml', 'baseline_year': '1997', 'cd4_nadir_before_tce': '140.0', 'baseline_log_rna': '5.1', 'baseline_cd4': '380.0', 'followup_log_rna': '5.3', 'followup_cd4': '460.0', 'baseline_mutation_count': '19', 'past_regimen_key': '2', 'new_regimen_key': '1', 'time_key': '1'}, {'tce_id': '2', 'xml_filename': '10.xml', 'baseline_year': '1998', 'cd4_nadir_before_tce': '401.0', 'baseline_log_rna': '4.0', 'baseline_cd4': '790.0', 'followup_log_rna': '3.4', 'followup_cd4': '641.0', 'baseline_mutation_count': '26', 'past_regimen_key': '4', 'new_regimen_key': '3', 'time_key': '2'}, {'tce_id': '3', 'xml_filename': '100.xml', 'baseline_year': '2000', 'cd4_nadir_before_tce': '101.0', 'baseline_log_rna': '4.4', 'baseline_cd4': '105.0', 'followup_log_rna': '3.2', 'followup_cd4': '157.0', 'baseline_mutation_count': '35', 'past_regimen_key': '6', 'new_regimen_key': '5', 'time_key': '4'}, {'tce_id': '4', 'xml_filename': '1000.xml', 'baseline_year': '2001', 'cd4_nadir_before_

In [37]:
rows = run_athena_query("""
    SELECT *
    FROM hivdb_tce.fact_tce
    LIMIT 10
""")

print(rows[:5])

[{'tce_id': '1', 'xml_filename': '1.xml', 'baseline_year': '1997', 'cd4_nadir_before_tce': '140.0', 'baseline_log_rna': '5.1', 'baseline_cd4': '380.0', 'followup_log_rna': '5.3', 'followup_cd4': '460.0', 'baseline_mutation_count': '19', 'past_regimen_key': '2', 'new_regimen_key': '1', 'time_key': '1'}, {'tce_id': '2', 'xml_filename': '10.xml', 'baseline_year': '1998', 'cd4_nadir_before_tce': '401.0', 'baseline_log_rna': '4.0', 'baseline_cd4': '790.0', 'followup_log_rna': '3.4', 'followup_cd4': '641.0', 'baseline_mutation_count': '26', 'past_regimen_key': '4', 'new_regimen_key': '3', 'time_key': '2'}, {'tce_id': '3', 'xml_filename': '100.xml', 'baseline_year': '2000', 'cd4_nadir_before_tce': '101.0', 'baseline_log_rna': '4.4', 'baseline_cd4': '105.0', 'followup_log_rna': '3.2', 'followup_cd4': '157.0', 'baseline_mutation_count': '35', 'past_regimen_key': '6', 'new_regimen_key': '5', 'time_key': '4'}, {'tce_id': '4', 'xml_filename': '1000.xml', 'baseline_year': '2001', 'cd4_nadir_before_

Dashboard.

In [38]:
rows = run_athena_query("""
    SELECT *
    FROM hivdb_tce.dashboard
    LIMIT 10
""")

print(rows[:5])

[{'tce_id': '1', 'baseline_year': '1997', 'baseline_log_rna': '5.1', 'regimen_group': 'PI'}, {'tce_id': '2', 'baseline_year': '1998', 'baseline_log_rna': '4.0', 'regimen_group': 'PI'}, {'tce_id': '3', 'baseline_year': '2000', 'baseline_log_rna': '4.4', 'regimen_group': 'PI'}, {'tce_id': '4', 'baseline_year': '2001', 'baseline_log_rna': '4.6', 'regimen_group': 'PI'}, {'tce_id': '5', 'baseline_year': '2007', 'baseline_log_rna': '3.8', 'regimen_group': 'PI'}]


How many regimens are there?

In [53]:
rows = run_athena_query("""
    SELECT regimen_signature, COUNT(*) as n
    FROM dashboard
    GROUP BY regimen_signature
    ORDER BY n DESC
""")

print(f"Total unique regimens: {len(rows)}")

Total unique regimens: 25


In [54]:
print(f"Regimens appearing 5+ times: {len([r for r in rows if int(r['n']) >= 5])}")

Regimens appearing 5+ times: 25


In [55]:
print(f"Regimens appearing 10+ times: {len([r for r in rows if int(r['n']) >= 10])}")

Regimens appearing 10+ times: 25
